# 🤖 KI-Integration in der Produktionsplanung

**Intelligente Produktionsplanung und KI**  
Universität Stuttgart  
Master Maschinenbau / Technologiemanagement

---

## Lernziele

Nach Bearbeitung dieses Notebooks können Sie:

1. Eine **End-to-End Machine Learning Pipeline** für Produktionsdaten aufbauen
2. **Supervised Learning Modelle** (Random Forest, Gradient Boosting) anwenden und bewerten
3. **Feature Importance** als prädiktive Relevanz interpretieren und mögliche Ansatzpunkte für weitere Untersuchungen identifizieren
4. **Szenario-Analysen** mit trainierten Modellen durchführen
5. **Industrie 4.0 Trends** und deren KI-Anwendungen einordnen
6. Eine **Implementierungsstrategie** für KI in der Produktion entwickeln

---

Das Notebook verwendet einen synthetischen Lehrdatensatz mit bewusst vorgegebenen Wirkungsbeziehungen. Diese gelten innerhalb des Datengenerators und sind nicht als allgemeine Produktionsgesetze zu verstehen.

## 1. Einführung: KI in der Produktionsplanung

### Warum KI in der Produktion?

Die **vierte industrielle Revolution (Industrie 4.0)** transformiert die Fertigungsindustrie durch:

| Treiber | Beschreibung |
|---------|-------------|
| **Datenverfügbarkeit** | Sensoren und betriebliche Systeme stellen umfangreiche Produktionsdaten bereit |
| **Rechenleistung** | Cloud Computing und GPUs ermöglichen komplexe Modelle |
| **Algorithmen** | Fortschritte in Machine Learning und Deep Learning |
| **Wettbewerb** | Kostendruck und Qualitätsanforderungen steigen |

### KI-Anwendungsfelder in der Produktion

```
┌─────────────────────────────────────────────────────────────────┐
│                    KI in der Produktion                         │
├─────────────────┬─────────────────┬─────────────────────────────┤
│   Prognose      │   Optimierung   │      Automatisierung        │
├─────────────────┼─────────────────┼─────────────────────────────┤
│ • Nachfrage     │ • Scheduling    │ • Qualitätskontrolle        │
│ • Qualität      │ • Ressourcen    │ • Predictive Maintenance    │
│ • Ausfälle      │ • Energie       │ • Robotersteuerung          │
│ • Kosten        │ • Routen        │ • Anomalieerkennung         │
└─────────────────┴─────────────────┴─────────────────────────────┘
```

### Machine Learning Grundkonzepte

| Lerntyp | Beschreibung | Beispiel in Produktion |
|---------|-------------|------------------------|
| **Supervised** | Lernen aus gelabelten Daten | Qualitätsprognose aus Prozessparametern |
| **Unsupervised** | Muster in ungelabelten Daten | Anomalieerkennung in Sensordaten |
| **Reinforcement** | Lernen durch Belohnung | Robotersteuerung, Scheduling |

In [ ]:
# ===================================================================
# Imports und Konfiguration
# ===================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    GridSearchCV
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Alle Bibliotheken erfolgreich geladen!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("Vorverarbeitung: One-Hot-Encoding innerhalb der ML-Pipeline")

---

## 2. Fallbeispiel: Intelligente Fertigungssteuerung

### Problemstellung

Ein Fertigungsunternehmen möchte datenbasierte Modelle einsetzen, um:

1. **Fertigungszeiten vorherzusagen** und die Kapazitätsplanung zu unterstützen
2. **Qualität zu prognostizieren** und Ansatzpunkte für die Qualitätsverbesserung zu untersuchen
3. **Energieverbrauch vorherzusagen** und alternative Prozesskonfigurationen zu vergleichen

### Datenbasis

Wir generieren einen **synthetischen Produktionsdatensatz** mit folgenden
Eingangsmerkmalen:

| Feature-Kategorie | Features | Beschreibung |
|-------------------|----------|-------------|
| **Produkt** | Produkttyp | Elektronik, Mechanik, Optik oder Hybrid |
| **Prozess** | Maschine, Schicht | Zugeordnete Maschine und Produktionsschicht |
| **Umgebung** | Temperatur_C, Luftfeuchtigkeit_pct | Umgebungsbedingungen |
| **Material** | Materialqualitaet | Synthetischer Qualitätsindex des Eingangsmaterials |
| **Personal** | Operator_Erfahrung_Jahre | Erfahrung in Jahren; vereinfachtes Merkmal, keine vollständige Beschreibung der Qualifikation |
| **Wartung** | Tage_seit_Wartung | Zeit seit der letzten Wartung; vereinfachtes Merkmal, keine vollständige Beschreibung des Maschinenzustands |

### Zielgrößen

Der Generator erzeugt vier Zielgrößen:

- **Fertigungszeit_min**
- **Qualitaetsscore**
- **Energieverbrauch_kWh**
- **Ausschussrate**

Im folgenden ML-Workflow werden **Fertigungszeit, Qualitätsscore und
Energieverbrauch** modelliert. Die Ausschussrate wird zusätzlich erzeugt,
aber hier nicht als eigene Zielgröße trainiert.

### Vorgegebene Zusammenhänge im Lehrdatensatz

- Die **Fertigungszeit** hängt von Produkttyp, Maschine, Schicht,
  Temperatur und Zeit seit Wartung ab.
- Der **Qualitätsscore** hängt von Materialqualität, Erfahrung,
  Zeit seit Wartung und Schicht ab. Der Erfahrungszuschlag steigt
  bis 15 Jahre und bleibt danach konstant.
- Der **Energieverbrauch** hängt von Produkttyp, Maschine und
  generierter Fertigungszeit ab.
- Die **Ausschussrate** hängt unter anderem vom Qualitätsscore
  und der Zeit seit Wartung ab.
- **Luftfeuchtigkeit** hat im Generator keinen direkten Einfluss
  auf die Zielgrößen. Sie dient als zusätzliches Merkmal, an dem
  mögliche zufällige Modellzusammenhänge diskutiert werden können.

**Wichtig:** Diese Beziehungen sind bewusst im Datengenerator vorgegeben.
Sie sind keine empirisch nachgewiesenen Produktionsgesetze.
Ein trainiertes Modell kann sie nur näherungsweise lernen und zusätzlich
zufällige Zusammenhänge aufgreifen.

In [ ]:
# ===================================================================
# Datensatz für KI-Anwendungen generieren
# ===================================================================

def generiere_produktionsdaten(n_samples=2000, seed=42):
    """
    Generiert einen synthetischen Lehrdatensatz mit bewusst vorgegebenen
Wirkungsbeziehungen und zufälligen Schwankungen.
    
    Parameter:
    ----------
    n_samples : int
        Anzahl der Datenpunkte
    seed : int
        Random Seed für Reproduzierbarkeit
        
    Returns:
    --------
    DataFrame : Produktionsdatensatz
    """
    np.random.seed(seed)
    
    # Kategorische Variablen
    produkttypen = ['Elektronik', 'Mechanik', 'Optik', 'Hybrid']
    maschinen = ['CNC_A', 'CNC_B', 'Robot_1', 'Robot_2', '3D_Drucker']
    schichten = ['Früh', 'Spät', 'Nacht']
    
    # Faktoren für vorgegebene Beziehungen im synthetischen Datengenerator
    komplexitaet = {'Elektronik': 1.2, 'Mechanik': 1.0, 'Optik': 1.6, 'Hybrid': 1.4}
    maschinen_effizienz = {'CNC_A': 1.1, 'CNC_B': 1.0, 'Robot_1': 1.3, 'Robot_2': 1.2, '3D_Drucker': 0.9}
    schicht_faktor = {'Früh': 1.0, 'Spät': 0.95, 'Nacht': 0.88}
    energie_faktor = {'CNC_A': 0.8, 'CNC_B': 1.0, 'Robot_1': 0.6, 'Robot_2': 0.7, '3D_Drucker': 1.2}
    
    daten = []
    
    for i in range(n_samples):
        # Zufällige Auswahl kategorischer Variablen
        produkttyp = np.random.choice(produkttypen)
        maschine = np.random.choice(maschinen)
        schicht = np.random.choice(schichten)
        
        # Numerische Features
        temperatur = np.random.normal(22, 2.5)
        luftfeuchtigkeit = np.random.normal(45, 8)
        materialqualitaet = np.random.uniform(6.5, 10)
        operator_erfahrung = np.random.exponential(4) + 0.5
        tage_seit_wartung = np.random.poisson(12)
        
        # === Zielgrößen mit vorgegebenen Beziehungen und Zufallskomponenten ===
        
        # 1. Fertigungszeit (Minuten)
        basis_zeit = 45 * komplexitaet[produkttyp]
        zeit = basis_zeit / maschinen_effizienz[maschine] / schicht_faktor[schicht]
        zeit *= (1 + (temperatur - 22) * 0.008)  # Temperatureinfluss
        zeit *= (1 + tage_seit_wartung * 0.004)  # Wartungseinfluss
        zeit *= np.random.normal(1, 0.08)  # Zufallsvariation
        fertigungszeit = max(zeit, 15)
        
        # 2. Qualitätsscore (0-1)
        qualitaet = 0.92
        qualitaet += (materialqualitaet - 8) * 0.02  # Materialeinfluss
        qualitaet += 0.05 * min(operator_erfahrung / 15, 1)  # Erfahrungseinfluss, Saettigung nach 15 Jahren
        qualitaet -= tage_seit_wartung * 0.002  # Wartungseinfluss
        qualitaet *= schicht_faktor[schicht]  # Schichteinfluss
        qualitaet += np.random.normal(0, 0.03)
        qualitaetsscore = np.clip(qualitaet, 0.70, 0.99)
        
        # 3. Energieverbrauch (kWh)
        basis_energie = 4 * komplexitaet[produkttyp]
        energie = basis_energie * energie_faktor[maschine] * (fertigungszeit / 60)
        energie *= np.random.normal(1, 0.1)
        energieverbrauch = max(energie, 0.5)
        
        # 4. Ausschusswahrscheinlichkeit
        ausschuss = 0.02 + (1 - qualitaetsscore) * 0.15
        ausschuss += tage_seit_wartung * 0.001
        ausschuss = np.clip(ausschuss + np.random.exponential(0.01), 0, 0.20)
        
        daten.append({
            'Produkttyp': produkttyp,
            'Maschine': maschine,
            'Schicht': schicht,
            'Temperatur_C': round(temperatur, 1),
            'Luftfeuchtigkeit_pct': round(luftfeuchtigkeit, 1),
            'Materialqualitaet': round(materialqualitaet, 1),
            'Operator_Erfahrung_Jahre': round(min(operator_erfahrung, 25), 1),
            'Tage_seit_Wartung': int(tage_seit_wartung),
            'Fertigungszeit_min': round(fertigungszeit, 1),
            'Qualitaetsscore': round(qualitaetsscore, 3),
            'Energieverbrauch_kWh': round(energieverbrauch, 2),
            'Ausschussrate': round(ausschuss, 4)
        })
    
    return pd.DataFrame(daten)

# Datensatz generieren
df = generiere_produktionsdaten(n_samples=2000)

print("📊 PRODUKTIONSDATENSATZ")
print("=" * 65)
print(f"\nAnzahl Datenpunkte: {len(df)}")
print(f"Anzahl Spalten: {len(df.columns)}")
print("Davon: 8 Eingangsmerkmale und 4 generierte Zielgrößen")
print("Im ML-Workflow modelliert: Fertigungszeit, Qualität und Energie")

print("\n📋 FEATURE-ÜBERSICHT:")
print("-" * 65)
print("Kategorische Features:")
for col in ['Produkttyp', 'Maschine', 'Schicht']:
    print(f"   {col}: {df[col].unique().tolist()}")

print("\nNumerische Features:")
num_cols = ['Temperatur_C', 'Luftfeuchtigkeit_pct', 'Materialqualitaet', 
           'Operator_Erfahrung_Jahre', 'Tage_seit_Wartung']
print(df[num_cols].describe().round(2).to_string())

print("\n🎯 ZIELGRÖSSEN:")
ziel_cols = ['Fertigungszeit_min', 'Qualitaetsscore', 'Energieverbrauch_kWh', 'Ausschussrate']
print(df[ziel_cols].describe().round(3).to_string())

In [ ]:
# ===================================================================
# Explorative Datenanalyse
# ===================================================================

def explorative_analyse(df):
    """
    Führt eine explorative Datenanalyse durch.
    """
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # 1. Fertigungszeit nach Produkttyp
    ax1 = axes[0, 0]
    df.boxplot(column='Fertigungszeit_min', by='Produkttyp', ax=ax1)
    ax1.set_title('Fertigungszeit nach Produkttyp')
    ax1.set_xlabel('Produkttyp')
    ax1.set_ylabel('Fertigungszeit (min)')
    plt.suptitle('')  # Entferne automatischen Titel
    
    # 2. Qualität nach Maschine
    ax2 = axes[0, 1]
    df.boxplot(column='Qualitaetsscore', by='Maschine', ax=ax2)
    ax2.set_title('Qualitätsscore nach Maschine')
    ax2.set_xlabel('Maschine')
    ax2.set_ylabel('Qualitätsscore')
    ax2.tick_params(axis='x', rotation=45)
    
    # 3. Energieverbrauch nach Schicht
    ax3 = axes[0, 2]
    df.boxplot(column='Energieverbrauch_kWh', by='Schicht', ax=ax3)
    ax3.set_title('Energieverbrauch nach Schicht')
    ax3.set_xlabel('Schicht')
    ax3.set_ylabel('Energieverbrauch (kWh)')
    
    # 4. Korrelation: Wartung vs. Qualität
    ax4 = axes[1, 0]
    ax4.scatter(df['Tage_seit_Wartung'], df['Qualitaetsscore'], alpha=0.3, s=20)
    z = np.polyfit(df['Tage_seit_Wartung'], df['Qualitaetsscore'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['Tage_seit_Wartung'].min(), df['Tage_seit_Wartung'].max(), 100)
    ax4.plot(x_line, p(x_line), 'r-', linewidth=2, label='Trendlinie')
    ax4.set_xlabel('Tage seit Wartung')
    ax4.set_ylabel('Qualitätsscore')
    ax4.set_title('Zusammenhang: Zeit seit Wartung und Qualität')
    ax4.legend()
    
    # 5. Korrelation: Erfahrung vs. Qualität
    ax5 = axes[1, 1]
    ax5.scatter(df['Operator_Erfahrung_Jahre'], df['Qualitaetsscore'], alpha=0.3, s=20)
    z = np.polyfit(df['Operator_Erfahrung_Jahre'], df['Qualitaetsscore'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, df['Operator_Erfahrung_Jahre'].max(), 100)
    ax5.plot(x_line, p(x_line), 'r-', linewidth=2, label='Trendlinie')
    ax5.set_xlabel('Operator Erfahrung (Jahre)')
    ax5.set_ylabel('Qualitätsscore')
    ax5.set_title('Zusammenhang: Erfahrung und Qualität')
    ax5.legend()
    
    # 6. Korrelationsmatrix (numerische Features)
    ax6 = axes[1, 2]
    korr_cols = ['Temperatur_C', 'Materialqualitaet', 'Operator_Erfahrung_Jahre', 
                'Tage_seit_Wartung', 'Fertigungszeit_min', 'Qualitaetsscore']
    korr_matrix = df[korr_cols].corr()
    im = ax6.imshow(korr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    ax6.set_xticks(range(len(korr_cols)))
    ax6.set_yticks(range(len(korr_cols)))
    ax6.set_xticklabels([c.replace('_', '\n') for c in korr_cols], fontsize=8)
    ax6.set_yticklabels([c.replace('_', '\n') for c in korr_cols], fontsize=8)
    ax6.set_title('Korrelationsmatrix')
    plt.colorbar(im, ax=ax6, shrink=0.8)
    
    plt.tight_layout()
    plt.show()

print("📊 EXPLORATIVE DATENANALYSE EDA")
print("=" * 50)
explorative_analyse(df)

print("\n💡 EINORDNUNG DER EXPLORATIVEN DATENANALYSE EDA:")
print("   • Der Generator gibt unterschiedliche Fertigungszeiten nach Produkttyp vor.")
print("   • Ein direkter Maschineneffekt auf Qualität wurde nicht vorgegeben.")
print("   • Der Generator enthält einen negativen Wartungsabstandseffekt auf Qualität.")
print("   • Der Erfahrungszuschlag steigt bis 15 Jahre und bleibt danach konstant.")
print("   • Die Diagramme zeigen beobachtete Zusammenhänge mit zufälliger Streuung.")
print("   • Bei realen Daten belegt ein solcher Zusammenhang allein keine Kausalität.")

---

## 3. Machine Learning Pipeline

### Überblick ML-Workflow

1. **Eingangsmerkmale und Zielgröße festlegen**
2. **Trainings- und Testdaten trennen**
3. **Vorverarbeitung und Modell in einer Pipeline zusammenfassen**
4. **Modelle durch Cross-Validation auf den Trainingsdaten vergleichen**
5. **Modell anhand des mittleren CV-MAE auswählen**
6. **Ausgewähltes Modell auf den Trainingsdaten fitten**
7. **Abschließend auf den zurückgehaltenen Testdaten bewerten**
8. **Vorhersagen und Modellverhalten analysieren**

### Vorverarbeitung

- Nominale Merkmale wie Produkttyp, Maschine und Schicht werden
  mit **One-Hot-Encoding** verarbeitet.
- Der Encoder wird innerhalb der Pipeline angepasst. Bei der
  Cross-Validation geschieht dies jeweils nur auf dem Trainingsfold.
- Für die hier verwendeten Baumverfahren ist keine Standardisierung
  der numerischen Merkmale erforderlich.
- Die gleiche trainierte Vorverarbeitung wird bei neuen Daten verwendet.

### Verwendete Algorithmen

| Algorithmus | Beschreibung | Einordnung |
|-------------|-------------|------------|
| **Random Forest** | Ensemble aus mehreren Entscheidungsbäumen | Flexibles Modell; Güte hängt von Daten und Parametern ab |
| **Gradient Boosting** | Sequenzielles Ensemble zur schrittweisen Fehlerreduktion | Kann hohe Prognosegüte erzielen; sorgfältige Validierung erforderlich |

**Keine Methode ist grundsätzlich überlegen.** Die Auswahl erfolgt
anhand des CV-MAE, nicht anhand des Testfehlers.

### Annahme zum Datensplit

Der zufällige Split ist hier für die unabhängig generierten synthetischen
Beobachtungen geeignet. Bei realen Produktionsdaten können zeitliche
Abhängigkeiten, Chargen oder wiederholte Messungen derselben Maschine
einen zeitlichen oder gruppierten Split erfordern.

In [ ]:
# ===================================================================
# Machine Learning Pipeline für Produktionsdaten
# ===================================================================

class ProduktionsML:
    """
    ML-Workflow mit One-Hot-Encoding innerhalb der Pipeline.

    Modellauswahl: mittlerer CV-MAE auf Trainingsdaten.
    Abschließende Bewertung: ausgewähltes Modell auf Testdaten.
    """

    def __init__(self, df):
        self.df = df.copy()

        self.feature_cols = [
            'Produkttyp',
            'Maschine',
            'Schicht',
            'Temperatur_C',
            'Luftfeuchtigkeit_pct',
            'Materialqualitaet',
            'Operator_Erfahrung_Jahre',
            'Tage_seit_Wartung'
        ]

        self.cat_cols = [
            'Produkttyp',
            'Maschine',
            'Schicht'
        ]

        self.num_cols = [
            col for col in self.feature_cols
            if col not in self.cat_cols
        ]

        self.models = {}
        self.model_names = {}
        self.splits = {}

    def prepare_features(self):
        """Gibt rohe Eingangsmerkmale zurück; kein Fit vor dem Split."""
        return self.df[self.feature_cols].copy()

    def make_pipeline(self, estimator):
        """Erstellt eine neue Pipeline für einen Regressor."""
        preprocessing = ColumnTransformer(
            transformers=[
                (
                    'kategorien',
                    OneHotEncoder(
                        handle_unknown='ignore',
                        sparse_output=False
                    ),
                    self.cat_cols
                ),
                (
                    'numerisch',
                    'passthrough',
                    self.num_cols
                )
            ]
        )

        return Pipeline([
            ('preprocessing', preprocessing),
            ('regressor', estimator)
        ])

    def train_and_evaluate(self, target_col, test_size=0.2):
        """
        Vergleicht Modelle per CV-MAE und bewertet nur das ausgewählte
        Modell abschließend auf den zurückgehaltenen Testdaten.
        """
        print("\n" + "=" * 65)
        print(f"🎯 MODELLTRAINING: {target_col}")
        print("=" * 65)

        X = self.prepare_features()
        y = self.df[target_col].copy()

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=42
        )

        self.splits[target_col] = {
            'X_train': X_train.copy(),
            'X_test': X_test.copy(),
            'y_train': y_train.copy(),
            'y_test': y_test.copy()
        }

        print(f"Training: {len(X_train)} Beobachtungen")
        print(f"Test:     {len(X_test)} Beobachtungen")

        estimators = {
            'Random Forest': RandomForestRegressor(
                n_estimators=100,
                max_depth=15,
                min_samples_split=5,
                random_state=42,
                n_jobs=1
            ),
            'Gradient Boosting': GradientBoostingRegressor(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=42
            )
        }

        cv = KFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        results = {}

        print("\n📈 MODELLVERGLEICH AUF TRAININGSDATEN")
        print(f"{'Modell':<20} {'CV-MAE':>12} {'Fold-Streuung':>16}")
        print("-" * 52)

        for name, estimator in estimators.items():
            pipeline = self.make_pipeline(estimator)

            cv_scores = cross_val_score(
                pipeline,
                X_train,
                y_train,
                cv=cv,
                scoring='neg_mean_absolute_error',
                n_jobs=1
            )

            fold_mae = -cv_scores
            cv_mae = fold_mae.mean()
            cv_std = fold_mae.std(ddof=1)

            results[name] = {
                'model': pipeline,
                'cv_mae': cv_mae,
                'cv_std': cv_std
            }

            print(
                f"{name:<20} {cv_mae:>12.4f} "
                f"{cv_std:>16.4f}"
            )

        best_model_name = min(
            results,
            key=lambda name: results[name]['cv_mae']
        )

        best = results[best_model_name]
        best_pipeline = best['model']

        # Erst nach der Auswahl auf allen Trainingsdaten fitten.
        best_pipeline.fit(X_train, y_train)

        # Abschließende Testbewertung des ausgewählten Modells.
        y_pred = best_pipeline.predict(X_test)

        best.update({
            'mae': mean_absolute_error(y_test, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
            'r2': r2_score(y_test, y_pred),
            'y_test': y_test,
            'y_pred': y_pred
        })

        # Importance je transformiertem Feature.
        transformed_names = (
            best_pipeline.named_steps['preprocessing']
            .get_feature_names_out()
        )

        importance = (
            best_pipeline.named_steps['regressor']
            .feature_importances_
        )

        best['feature_names'] = transformed_names
        best['feature_importance'] = importance

        # One-Hot-Importances auf ursprüngliche Merkmale aggregieren.
        encoder = (
            best_pipeline.named_steps['preprocessing']
            .named_transformers_['kategorien']
        )

        aggregated = {}
        position = 0

        for col, categories in zip(
            self.cat_cols,
            encoder.categories_
        ):
            count = len(categories)
            aggregated[col] = float(
                importance[position:position + count].sum()
            )
            position += count

        for col in self.num_cols:
            aggregated[col] = float(importance[position])
            position += 1

        best['importance_original'] = (
            pd.Series(aggregated)
            .sort_values(ascending=False)
        )

        self.models[target_col] = best_pipeline
        self.model_names[target_col] = best_model_name

        print(f"\n🏆 Nach CV-MAE ausgewählt: {best_model_name}")
        print(f"CV-MAE:   {best['cv_mae']:.4f}")
        print("\nABSCHLIESSENDE TESTBEWERTUNG")
        print(f"Test-MAE:  {best['mae']:.4f}")
        print(f"Test-RMSE: {best['rmse']:.4f}")
        print(f"Test-R²:   {best['r2']:.4f}")

        print(
            "\nDie Fold-Streuung ist die Streuung der fünf Fold-MAE, "
            "kein Konfidenzintervall."
        )

        return results, best_model_name

    def plot_results(self, results, target_col, best_model_name):
        """Visualisiert die Testresultate des ausgewählten Modells."""
        best = results[best_model_name]

        actual = np.asarray(best['y_test'], dtype=float)
        predicted = np.asarray(best['y_pred'], dtype=float)
        residuals = actual - predicted

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        ax = axes[0, 0]
        ax.scatter(actual, predicted, alpha=0.5, s=30)

        limits = [
            min(actual.min(), predicted.min()),
            max(actual.max(), predicted.max())
        ]

        ax.plot(
            limits,
            limits,
            'r--',
            linewidth=2,
            label='Perfekte Vorhersage'
        )
        ax.set_xlabel('Tatsächliche Werte')
        ax.set_ylabel('Vorhergesagte Werte')
        ax.set_title(f'{best_model_name}: Testvorhersagen')
        ax.legend()
        ax.text(
            0.05,
            0.95,
            f"R² = {best['r2']:.3f}\nMAE = {best['mae']:.3f}",
            transform=ax.transAxes,
            va='top',
            bbox=dict(
                boxstyle='round',
                facecolor='white',
                alpha=0.8
            )
        )

        ax = axes[0, 1]
        ax.scatter(predicted, residuals, alpha=0.5, s=30)
        ax.axhline(0, color='red', linestyle='--')
        ax.set_xlabel('Vorhergesagte Werte')
        ax.set_ylabel('Residuen: Ist minus Vorhersage')
        ax.set_title('Residuen auf Testdaten')

        ax = axes[1, 0]
        importance = (
            best['importance_original']
            .sort_values(ascending=True)
        )
        ax.barh(
            importance.index,
            importance.values,
            color='steelblue'
        )
        ax.set_xlabel('Aggregierte impurity-basierte Importance')
        ax.set_title('Importance der ursprünglichen Merkmale')

        ax = axes[1, 1]
        ax.hist(
            residuals,
            bins=30,
            edgecolor='black',
            alpha=0.7
        )
        ax.axvline(0, color='red', linestyle='--')
        ax.set_xlabel('Residuen')
        ax.set_ylabel('Häufigkeit')
        ax.set_title('Fehlerverteilung auf Testdaten')

        fig.suptitle(
            f'Modellevaluation: {target_col}',
            fontsize=14,
            fontweight='bold'
        )
        plt.tight_layout()
        plt.show()

        print(f"\n📊 FEATURE IMPORTANCE: {target_col}")
        print(
            best['importance_original']
            .to_string(float_format=lambda value: f"{value:.3f}")
        )
        print(
            "\nOne-Hot-Spalten wurden je ursprünglichem Merkmal "
            "zusammengefasst."
        )
        print(
            "Importance beschreibt die Nutzung durch dieses Modell, "
            "nicht die Richtung oder Kausalität eines Effekts."
        )
        print(
            "Impurity-basierte Importance kann unter anderem durch "
            "Merkmalsstruktur und Korrelationen verzerrt werden."
        )

    def predict(self, target_col, input_data):
        """Prognostiziert mit derselben trainierten Vorverarbeitung."""
        if target_col not in self.models:
            raise ValueError(
                f"Kein trainiertes Modell für {target_col}."
            )

        missing = [
            col for col in self.feature_cols
            if col not in input_data
        ]
        if missing:
            raise ValueError(f"Fehlende Merkmale: {missing}")

        # Neue Kategorien nicht stillschweigend für Szenarien zulassen.
        encoder = (
            self.models[target_col]
            .named_steps['preprocessing']
            .named_transformers_['kategorien']
        )

        for col, categories in zip(
            self.cat_cols,
            encoder.categories_
        ):
            if input_data[col] not in categories:
                raise ValueError(
                    f"Unbekannte Kategorie für {col}: "
                    f"{input_data[col]}"
                )

        X_new = pd.DataFrame([input_data])[self.feature_cols]
        return float(self.models[target_col].predict(X_new)[0])


ml_pipeline = ProduktionsML(df)
print("✅ ML-Pipeline initialisiert")

---

## 4. Modelltraining und Evaluation

Wir trainieren Modelle für drei Zielgrößen:
1. **Fertigungszeit** - für Kapazitätsplanung
2. **Qualitätsscore** - für Qualitätsmanagement
3. **Energieverbrauch** - für Nachhaltigkeitsoptimierung

### Auswahl und Bewertung

Die Modellauswahl erfolgt anhand des mittleren MAE aus fünf
Cross-Validation-Folds auf den Trainingsdaten.

Nur das ausgewählte Modell wird anschließend auf den zurückgehaltenen
Testdaten bewertet. Die Testwerte dürfen nicht zur nachträglichen
Auswahl von Parametern oder Merkmalen verwendet werden.

Die angezeigte Feature Importance ist impurity-basiert.
One-Hot-Spalten werden für die Darstellung auf die ursprünglichen
Merkmale aggregiert. Importance zeigt weder die Richtung eines
Zusammenhangs noch einen kausalen Effekt.

**Hinweis:** Nach Änderungen an Datengenerator, Encoding oder
Modellauswahl müssen sämtliche Ergebnisse neu berechnet werden.


In [ ]:
# ===================================================================
# Modell 1: Fertigungszeit-Prognose
# ===================================================================

results_zeit, best_zeit = ml_pipeline.train_and_evaluate('Fertigungszeit_min')
ml_pipeline.plot_results(results_zeit, 'Fertigungszeit_min', best_zeit)

In [ ]:
# ===================================================================
# Modell 2: Qualitätsprognose
# ===================================================================

results_qual, best_qual = ml_pipeline.train_and_evaluate('Qualitaetsscore')
ml_pipeline.plot_results(results_qual, 'Qualitaetsscore', best_qual)

In [ ]:
# ===================================================================
# Modell 3: Energieverbrauch-Prognose
# ===================================================================

results_energie, best_energie = ml_pipeline.train_and_evaluate('Energieverbrauch_kWh')
ml_pipeline.plot_results(results_energie, 'Energieverbrauch_kWh', best_energie)

---

## 5. Modellbasierte Szenarioanalyse

Mit den trainierten Modellen vergleichen wir vorgegebene alternative
Eingangskonfigurationen.

Die Ergebnisse sind **Punktprognosen des Modells**, keine nachgewiesenen
Wirkungen realer Maßnahmen. Es wird kein Optimierungsproblem gelöst.

### Grenzen

- Die Modelle bilden gelernte statistische Zusammenhänge ab.
- Ein verändertes Merkmal muss nicht kausal die prognostizierte
  Veränderung erzeugen.
- Auch Kombinationen einzeln bekannter Merkmalswerte können im
  Training selten oder nicht vertreten gewesen sein.
- Kosten, Kapazitäten, technische Machbarkeit und Umsetzungsrisiken
  werden hier nicht bewertet.
- Kleine Unterschiede können im Bereich der Modellfehler liegen.
- Die getrennten Modelle für Zeit, Qualität und Energie garantieren
  keine vollständige physikalische Konsistenz ihrer Vorhersagen.

### Relative Veränderung

Wir verwenden für alle Zielgrößen:

$$
\Delta[\%] =
100 \cdot
\frac{\text{Szenariowert}-\text{Basiswert}}{\text{Basiswert}}
$$

Negative Werte bedeuten eine Verringerung, positive Werte eine Erhöhung.
Bei Zeit und Energie ist eine Verringerung hier erwünscht,
beim Qualitätsscore eine Erhöhung.


In [ ]:
# ===================================================================
# Modellbasierte Szenarioanalyse
# ===================================================================

def szenario_analyse(ml_pipeline):
    basis = {
        'Produkttyp': 'Elektronik',
        'Maschine': 'CNC_B',
        'Schicht': 'Früh',
        'Temperatur_C': 22.0,
        'Luftfeuchtigkeit_pct': 45.0,
        'Materialqualitaet': 7.5,
        'Operator_Erfahrung_Jahre': 3.0,
        'Tage_seit_Wartung': 15
    }

    niedriger_wartungsabstand = int(
        ml_pipeline.df['Tage_seit_Wartung'].min()
    )

    szenarien = {
        'Basis': basis.copy(),
        'Alternative Maschine': {
            **basis,
            'Maschine': 'Robot_1'
        },
        'Höhere Materialqualität': {
            **basis,
            'Materialqualitaet': 9.5
        },
        'Mehr Erfahrung': {
            **basis,
            'Operator_Erfahrung_Jahre': 10.0
        },
        'Kürzerer Wartungsabstand': {
            **basis,
            'Tage_seit_Wartung': niedriger_wartungsabstand
        },
        'Kombiniertes Szenario': {
            **basis,
            'Maschine': 'Robot_1',
            'Materialqualitaet': 9.5,
            'Operator_Erfahrung_Jahre': 10.0,
            'Tage_seit_Wartung': niedriger_wartungsabstand
        }
    }

    print("\n🔍 MODELLBASIERTE SZENARIOANALYSE")
    print(
        "Niedriger Wartungsabstand im Alternativszenario:",
        niedriger_wartungsabstand,
        "Tage"
    )

    rows = []

    for name, parameters in szenarien.items():
        # Einzelne numerische Werte gegen beobachtete Bereiche prüfen.
        for col in ml_pipeline.num_cols:
            minimum = ml_pipeline.df[col].min()
            maximum = ml_pipeline.df[col].max()
            value = parameters[col]

            if not minimum <= value <= maximum:
                print(
                    f"Hinweis: {name}, {col}={value} liegt "
                    f"außerhalb [{minimum}, {maximum}]."
                )

        rows.append({
            'Szenario': name,
            'Fertigungszeit_min': ml_pipeline.predict(
                'Fertigungszeit_min', parameters
            ),
            'Qualitaetsscore': ml_pipeline.predict(
                'Qualitaetsscore', parameters
            ),
            'Energieverbrauch_kWh': ml_pipeline.predict(
                'Energieverbrauch_kWh', parameters
            )
        })

    results = pd.DataFrame(rows)
    targets = [
        'Fertigungszeit_min',
        'Qualitaetsscore',
        'Energieverbrauch_kWh'
    ]

    baseline = results.loc[
        results['Szenario'] == 'Basis'
    ].iloc[0]

    for target in targets:
        results[f'Delta_{target}_pct'] = (
            100 * (results[target] - baseline[target])
            / baseline[target]
        )

    print("\nPUNKT­PROGNOSEN")
    print(
        results[['Szenario'] + targets]
        .round(3)
        .to_string(index=False)
    )

    delta_cols = [
        f'Delta_{target}_pct'
        for target in targets
    ]

    print("\nRELATIVE VERÄNDERUNG GEGENÜBER BASIS [%]")
    print(
        results[['Szenario'] + delta_cols]
        .round(2)
        .to_string(index=False)
    )

    fig, axes = plt.subplots(1, 3, figsize=(17, 6))

    labels = [
        'Fertigungszeit (min)',
        'Qualitätsscore',
        'Energieverbrauch (kWh)'
    ]

    x = np.arange(len(results))

    for ax, target, label in zip(axes, targets, labels):
        ax.bar(
            x,
            results[target],
            color='steelblue',
            alpha=0.8
        )
        ax.axhline(
            baseline[target],
            color='black',
            linestyle='--',
            label='Basis'
        )
        ax.set_xticks(x)
        ax.set_xticklabels(
            results['Szenario'],
            rotation=45,
            ha='right'
        )
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend()

    fig.suptitle(
        'Modellprognosen für vorgegebene Szenarien',
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    print(
        "\nKeine nachgewiesenen kausalen Wirkungen und "
        "keine betriebswirtschaftliche Optimierung."
    )
    print(
        "Die Bereichsprüfung einzelner Werte belegt nicht, "
        "dass deren Kombination ausreichend im Training vertreten ist."
    )
    print(
        "Kleine Prognoseunterschiede im Verhältnis zum Testfehler "
        "vorsichtig interpretieren."
    )

    return results


df_szenarien = szenario_analyse(ml_pipeline)

---

## 6. Industrie 4.0 und Zukunftsperspektiven

### Die Vision der Smart Factory

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         SMART FACTORY                                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐              │
│   │ Sensoren│──▶│  Edge   │───▶│  Cloud  │──▶│   KI    │              │
│   │  IoT    │    │Computing│    │Platform │    │ Modelle │              │
│   └─────────┘    └─────────┘    └─────────┘    └────┬────┘              │
│                                                      │                  │
│                                                      ▼                  │
│   ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐              │
│   │Maschinen│◀──│Steuerung│◀───│Optimier.│◀──│Vorhersage│             │
│   │ Aktoren │    │ SPS/PLC │    │         │    │Dashboard│              │
│   └─────────┘    └─────────┘    └─────────┘    └─────────┘              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ===================================================================
# Industrie 4.0 Trends und Zukunftsperspektiven
# ===================================================================

def zeige_industrie40_trends():
    """
    Zeigt Industrie 4.0 Trends und KI-Anwendungen.
    """
    
    print("\n🏭 INDUSTRIE 4.0 TRENDS UND KI-ANWENDUNGEN")
    print("=" * 70)
    
    trends = {
        'Digital Twin': {
            'icon': '🔄',
            'beschreibung': 'Digitale Zwillinge von Produktionsanlagen',
            'ki_methoden': 'Simulation, Reinforcement Learning, Zeitreihenanalyse',
            'nutzen': 'Virtuelle Erprobung vor realer Umsetzung; Modell- und Umsetzungsrisiken bleiben',
            'reifegrad': 3  # 1-5 Skala
        },
        'Predictive Maintenance': {
            'icon': '🔧',
            'beschreibung': 'Vorausschauende Wartung basierend auf Sensordaten',
            'ki_methoden': 'Anomalieerkennung, Survival Analysis, LSTM',
            'nutzen': 'Kann ungeplante Stillstände reduzieren; Nutzen muss fallspezifisch nachgewiesen werden',
            'reifegrad': 4
        },
        'Qualität 4.0': {
            'icon': '✅',
            'beschreibung': 'KI-basierte Qualitätskontrolle',
            'ki_methoden': 'Computer Vision, CNN, Defect Detection',
            'nutzen': 'Automatisierte Vollprüfung und zeitnahes Feedback; keine Garantie fehlerfreier Erkennung',
            'reifegrad': 4
        },
        'Adaptive Fertigung': {
            'icon': '⚙️',
            'beschreibung': 'Selbstoptimierende Produktionsprozesse',
            'ki_methoden': 'Reinforcement Learning, Bayesian Optimization',
            'nutzen': 'Automatische Prozessanpassung',
            'reifegrad': 2
        },
        'Intelligente Planung': {
            'icon': '📊',
            'beschreibung': 'KI-gestützte Produktions- und Kapazitätsplanung',
            'ki_methoden': 'Demand Forecasting, Constraint Optimization',
            'nutzen': 'Bessere Ressourcenauslastung, weniger Bestände',
            'reifegrad': 3
        },
        'Autonome Logistik': {
            'icon': '🤖',
            'beschreibung': 'Selbstfahrende Transportsysteme (AGVs)',
            'ki_methoden': 'Path Planning, SLAM, Multi-Agent RL',
            'nutzen': 'Flexible, skalierbare Intralogistik',
            'reifegrad': 3
        }
    }
    print(
        "\nDie folgenden Reifegrade sind illustrative Annahmen "
        "für einen fiktiven Lehrfall."
    )
    print(
        "Sie sind keine allgemeine Bewertung der Technologien "
        "und kein standardisiertes Reifegradmodell."
    )
    for name, details in trends.items():
        print(f"\n{details['icon']} {name}")
        print(f"   📝 {details['beschreibung']}")
        print(f"   🤖 KI-Methoden: {details['ki_methoden']}")
        print(f"   💡 Nutzen: {details['nutzen']}")
        print(f"   📈 Reifegrad: {'●' * details['reifegrad']}{'○' * (5-details['reifegrad'])}")
    
    # Visualisierung: Technologie-Roadmap
    fig, ax = plt.subplots(figsize=(14, 6))
    
    trend_namen = list(trends.keys())
    reifegrade = [trends[t]['reifegrad'] for t in trend_namen]
    
    colors = plt.cm.RdYlGn([r/5 for r in reifegrade])
    
    y_pos = np.arange(len(trend_namen))
    bars = ax.barh(y_pos, reifegrade, color=colors, edgecolor='black')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{trends[t]['icon']} {t}" for t in trend_namen])
    ax.set_xlabel('Illustrative Einordnung im Lehrfall (1–5)')
    ax.set_title('Beispielhafte Technologieeinordnung – keine allgemeine Reifegradbewertung', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 5.5)
    
    # Reifegrad-Labels
    for i, (bar, rg) in enumerate(zip(bars, reifegrade)):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
               f'{rg}/5', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return trends

trends = zeige_industrie40_trends()

In [ ]:
# ===================================================================
# Implementierungsstrategie
# ===================================================================

def zeige_implementierungsstrategie():
    """
    Zeigt eine praktische Implementierungsstrategie für KI in der Produktion.
    """
    
    print("\n🎯 IMPLEMENTIERUNGSSTRATEGIE: KI IN DER PRODUKTION")
    print("=" * 70)
    print(
        "\nBeispielhafte Vorgehensweise, kein verbindliches Phasenmodell."
    )
    print(
        "Zeiträume, Investitions- und Risikostufen sind illustrative "
        "Annahmen und hängen vom konkreten Use Case ab."
    )
    
    phasen = {
        'Phase 1: Foundation (0-6 Monate)': {
            'ziele': 'Grundlagen schaffen',
            'aktivitaeten': [
                'Dateninfrastruktur aufbauen (Data Lake)',
                'IoT-Sensoren an kritischen Anlagen installieren',
                'Datenqualität sicherstellen (Governance)',
                'Team aufbauen / Kompetenzen entwickeln',
                'Use Cases priorisieren (Quick Wins)'
            ],
            'investition': '€€',
            'risiko': 'Niedrig'
        },
        'Phase 2: Pilot (6-12 Monate)': {
            'ziele': 'Proof of Concept',
            'aktivitaeten': [
                'Erste ML-Modelle für ausgewählte Use Cases',
                'Dashboards für Datenvisualisierung',
                'Pilotprojekt: Predictive Quality',
                'ROI messen und dokumentieren',
                'Lessons Learned sammeln'
            ],
            'investition': '€€€',
            'risiko': 'Mittel'
        },
        'Phase 3: Scale (12-24 Monate)': {
            'ziele': 'Skalierung erfolgreicher Piloten',
            'aktivitaeten': [
                'MLOps-Plattform einführen',
                'Modelle in Produktion ausrollen',
                'Integration in bestehende Systeme (ERP, MES)',
                'Mitarbeiter schulen',
                'Weitere Use Cases erschließen'
            ],
            'investition': '€€€€',
            'risiko': 'Mittel-Hoch'
        },
        'Phase 4: Transform (24+ Monate)': {
            'ziele': 'Digitale Transformation',
            'aktivitaeten': [
                'KI als Kernkompetenz etablieren',
                'Autonome Optimierungssysteme',
                'Digitale Zwillinge für geeignete Anlagen und priorisierte Use Cases',
                'Neue Geschäftsmodelle (as-a-Service)',
                'Kontinuierliche Innovation - Kleine, überprüfbare Schritte statt Big Bang'
            ],
            'investition': '€€€€€',
            'risiko': 'Fallspezifisch zu bewerten'
        }
    }
    
    for phase, details in phasen.items():
        print(f"\n📍 {phase}")
        print(f"   🎯 Ziel: {details['ziele']}")
        print(f"   💰 Investition: {details['investition']}")
        print(f"   ⚠️ Risiko: {details['risiko']}")
        print(f"   📋 Aktivitäten:")
        for akt in details['aktivitaeten']:
            print(f"      • {akt}")
    
    # Erfolgsfaktoren
    print("\n" + "="*70)
    print("💡 KRITISCHE ERFOLGSFAKTOREN")
    print("=" * 70)
    
    erfolgsfaktoren = [
        ('Management Commitment', 'Top-Down Unterstützung und Vision'),
        ('Datenqualität', 'Garbage In = Garbage Out'),
        ('Change Management', 'Mitarbeiter mitnehmen, nicht ersetzen'),
        ('Schrittweise Vorgehen', 'Kleine Erfolge statt Big Bang'),
        ('Cross-funktionale Teams', 'IT + OT + Fachbereich'),
        ('Realistische Erwartungen', 'KI ist kein Allheilmittel'),
        ('Validierung und Freigabe', 'Modellgüte, Prozessrisiken und Einsatzgrenzen prüfen'),
        ('Monitoring', 'Datenänderungen und Modellverschlechterung überwachen'),
        ('Verantwortlichkeiten', 'Fachliche Prüfung und Eskalationswege festlegen'),
        ('Kontinuierliches Lernen', 'Modelle müssen gepflegt werden')
    ]
    
    for i, (faktor, erklaerung) in enumerate(erfolgsfaktoren, 1):
        print(f"   {i}. {faktor}")
        print(f"      → {erklaerung}")

zeige_implementierungsstrategie()

---

## 7. Aufgaben für Studierende

Bearbeiten Sie die folgenden Aufgaben, um Ihr Verständnis der KI-Integration in der Produktion zu vertiefen.

### ✏️ Aufgabe 1: Modellvarianten und Hyperparameter untersuchen

Verwenden Sie die neu berechneten Ergebnisse der Qualitätsprognose
als Ausgangspunkt.

1. Erstellen Sie zusätzliche Merkmale, beispielsweise Interaktionen
   zwischen Wartungsabstand, Materialqualität und Erfahrung.
2. Vergleichen Sie ein Random-Forest-Basismodell mit einer Variante,
   die zusätzliche Merkmale und abgestimmte Hyperparameter verwendet.
3. Wählen Sie Hyperparameter ausschließlich durch Cross-Validation
   auf den Trainingsdaten.
4. Vergleichen Sie die CV-Ergebnisse. Eine Verbesserung ist nicht garantiert.
5. Bewerten Sie die vorab festgelegten Varianten abschließend auf
   demselben Testdatensatz.

**Wichtig:** Die zusätzlichen Merkmale sind Hypothesen.
Bezeichnungen wie „kritischer Wartungsabstand“ oder „optimale Temperatur“
wären ohne entsprechende fachliche Begründung zu stark.

Die Testdaten dürfen nicht wiederholt zur Auswahl neuer Varianten
verwendet werden. Nach einer solchen Rückkopplung wäre für eine
unabhängige Bewertung ein neuer Testdatensatz erforderlich.

In [ ]:
# Ihre Lösung für Aufgabe 1:
# ===========================

# TODO: Feature Engineering implementieren
# TODO: Hyperparameter-Tuning durchführen
# TODO: Modelle vergleichen



In [ ]:
# ===================================================================
# LÖSUNG Aufgabe 1: Modellvarianten untersuchen
# ===================================================================

print("📝 AUFGABE 1: FEATURE ENGINEERING UND TUNING")

split = ml_pipeline.splits['Qualitaetsscore']

X_train_base = split['X_train'].copy()
X_test_base = split['X_test'].copy()
y_train = split['y_train'].copy()
y_test = split['y_test'].copy()


def add_features(X):
    """Deterministische Merkmale ohne Verwendung der Zielgröße."""
    enhanced = X.copy()

    enhanced['Wartung_x_Material'] = (
        enhanced['Tage_seit_Wartung']
        * enhanced['Materialqualitaet']
    )
    enhanced['Erfahrung_x_Material'] = (
        enhanced['Operator_Erfahrung_Jahre']
        * enhanced['Materialqualitaet']
    )
    enhanced['Temperatur_Abstand_22'] = (
        enhanced['Temperatur_C'] - 22
    ).abs()
    enhanced['Wartungsabstand_ueber_20'] = (
        enhanced['Tage_seit_Wartung'] > 20
    ).astype(int)

    return enhanced


X_train_enh = add_features(X_train_base)
X_test_enh = add_features(X_test_base)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline = ml_pipeline.make_pipeline(
    RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=1
    )
)

baseline_cv_mae = -cross_val_score(
    baseline,
    X_train_base,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=1
).mean()

cat_cols = ml_pipeline.cat_cols
num_cols_enh = [
    col for col in X_train_enh.columns
    if col not in cat_cols
]

enhanced_preprocessing = ColumnTransformer([
    (
        'kategorien',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ),
        cat_cols
    ),
    (
        'numerisch',
        'passthrough',
        num_cols_enh
    )
])

enhanced_pipeline = Pipeline([
    ('preprocessing', enhanced_preprocessing),
    (
        'regressor',
        RandomForestRegressor(
            random_state=42,
            n_jobs=1
        )
    )
])

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 15, 20],
    'regressor__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    enhanced_pipeline,
    param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=1,
    refit=True
)

grid_search.fit(X_train_enh, y_train)

print("\nBeste Parameter nach CV-MAE:")
print(grid_search.best_params_)
print(f"Basis-CV-MAE: {baseline_cv_mae:.4f}")
print(f"Bestes Tuning-CV-MAE: {-grid_search.best_score_:.4f}")
print(
    "Der beste Tuning-CV-Wert wurde unter mehreren Kandidaten "
    "ausgewählt und kann optimistisch sein."
)

# Vorab definierte Varianten abschließend auf demselben Testset.
baseline.fit(X_train_base, y_train)

pred_base = baseline.predict(X_test_base)
pred_enh = grid_search.best_estimator_.predict(X_test_enh)

comparison = pd.DataFrame([
    {
        'Variante': 'Random-Forest-Basis',
        'Test-MAE': mean_absolute_error(y_test, pred_base),
        'Test-RMSE': np.sqrt(
            mean_squared_error(y_test, pred_base)
        ),
        'Test-R2': r2_score(y_test, pred_base)
    },
    {
        'Variante': 'Feature Engineering + Tuning',
        'Test-MAE': mean_absolute_error(y_test, pred_enh),
        'Test-RMSE': np.sqrt(
            mean_squared_error(y_test, pred_enh)
        ),
        'Test-R2': r2_score(y_test, pred_enh)
    }
])

print("\nABSCHLIESSENDER VERGLEICH")
print(comparison.round(4).to_string(index=False))

delta_r2 = (
    comparison.loc[1, 'Test-R2']
    - comparison.loc[0, 'Test-R2']
)

mae_reduction_pct = (
    100
    * (
        comparison.loc[0, 'Test-MAE']
        - comparison.loc[1, 'Test-MAE']
    )
    / comparison.loc[0, 'Test-MAE']
)

print(f"\nR²-Differenz: {delta_r2:+.4f}")
print(f"Relative MAE-Reduktion: {mae_reduction_pct:+.2f} %")
print("Ein negativer Wert bedeutet eine Verschlechterung des MAE.")
print(
    "Feature Engineering und Tuning wurden gemeinsam verändert; "
    "ihre einzelnen Beiträge sind hier nicht isoliert."
)
print(
    "Diese Testresultate nicht zur wiederholten Auswahl weiterer "
    "Varianten verwenden."
)

### ✏️ Aufgabe 2: ROI-Berechnung

Berechnen Sie den Return on Investment (ROI) für die KI-Implementierung:

**Annahmen:**
- Jährliche Produktion: 100.000 Einheiten
- Kosten pro defekte Einheit: 50 €
- Aktuelle Ausschussrate: 4%
- Mit KI-Optimierung: 2.5% (Reduktion um 37.5%)
- KI-Implementierungskosten: 150.000 € (einmalig)
- Jährliche Wartungskosten: 30.000 €

**Berechnen Sie:**
1. Jährliche Einsparung durch Ausschussreduktion
2. ROI im ersten Jahr
3. Amortisationszeit (Payback Period)

**Einordnung:** Alle Zahlen sind Annahmen eines fiktiven Übungsfalls.
Die Ausschussreduktion ist kein Ergebnis der zuvor trainierten Modelle.

Wir definieren hier:

$$
ROI_{\text{Jahr 1}} =
\frac{\text{jährliche Nettoeinsparung}-\text{Anfangsinvestition}}
{\text{Anfangsinvestition}}
\cdot 100
$$

Die einfache Amortisationszeit ergibt sich aus Anfangsinvestition
geteilt durch jährliche Nettoeinsparung.

Die Rechnung berücksichtigt keine Abzinsung, Steuern, Anlaufphase,
Finanzierung oder zusätzlichen Umsetzungskosten. Sie setzt konstante
jährliche Nettoeinsparungen voraus.

In [ ]:
# Ihre Lösung für Aufgabe 2:
# ===========================

# TODO: Einsparung berechnen
# TODO: ROI berechnen
# TODO: Amortisationszeit berechnen



In [ ]:
# ===================================================================
# LÖSUNG Aufgabe 2 (zur Selbstkontrolle)
# ===================================================================

print("📝 LÖSUNG AUFGABE 2: ROI-Berechnung")
print("=" * 65)

# Parameter
produktion_jahr = 100000
kosten_defekt = 50
ausschuss_vorher = 0.04
ausschuss_nachher = 0.025
ki_investition = 150000
ki_wartung_jahr = 30000

print("\n📋 PARAMETER:")
print("-" * 40)
print(f"   Produktion/Jahr:      {produktion_jahr:>12,} Einheiten")
print(f"   Kosten/Defekt:        {kosten_defekt:>12} €")
print(f"   Ausschuss vorher:     {ausschuss_vorher*100:>12.1f}%")
print(f"   Ausschuss nachher:    {ausschuss_nachher*100:>12.1f}%")
print(f"   KI-Investition:       {ki_investition:>12,} €")
print(f"   KI-Wartung/Jahr:      {ki_wartung_jahr:>12,} €")

# Berechnungen
print("\n📊 BERECHNUNGEN:")
print("-" * 40)

# 1. Ausschusskosten vorher und nachher
defekte_vorher = produktion_jahr * ausschuss_vorher
defekte_nachher = produktion_jahr * ausschuss_nachher
kosten_vorher = defekte_vorher * kosten_defekt
kosten_nachher = defekte_nachher * kosten_defekt

print(f"   Defekte vorher:       {defekte_vorher:>12,.0f} Stück")
print(f"   Defekte nachher:      {defekte_nachher:>12,.0f} Stück")
print(f"   Einsparung Defekte:   {defekte_vorher - defekte_nachher:>12,.0f} Stück")

# 2. Jährliche Einsparung
brutto_einsparung = kosten_vorher - kosten_nachher
netto_einsparung = brutto_einsparung - ki_wartung_jahr

print(f"\n   Ausschusskosten vorher: {kosten_vorher:>10,.0f} €/Jahr")
print(f"   Ausschusskosten nachher: {kosten_nachher:>9,.0f} €/Jahr")
print(f"   Brutto-Einsparung:      {brutto_einsparung:>10,.0f} €/Jahr")
print(f"   - KI-Wartungskosten:    {ki_wartung_jahr:>10,.0f} €/Jahr")
print(f"   ─────────────────────────────────────")
print(f"   Netto-Einsparung:       {netto_einsparung:>10,.0f} €/Jahr")

# 3. ROI berechnen
print("\n📈 ROI-ANALYSE:")
print("-" * 40)

# ROI Jahr 1 (inkl. Investition)
gewinn_jahr1 = netto_einsparung - ki_investition
roi_jahr1 = gewinn_jahr1 / ki_investition * 100

print(f"   Jahr 1:")
print(f"      Netto-Einsparung:    {netto_einsparung:>10,.0f} €")
print(f"      - Investition:       {ki_investition:>10,.0f} €")
print(f"      = Netto-Cashflow:    {gewinn_jahr1:>+10,.0f} €")
print(f"      ROI Jahr 1:          {roi_jahr1:>+10.1f}%")

# Amortisationszeit
payback_monate = ki_investition / (netto_einsparung / 12)
payback_jahre = payback_monate / 12

print(f"\n   Amortisation:")
print(f"      Payback Period:      {payback_monate:>10.1f} Monate")
print(f"                           {payback_jahre:>10.1f} Jahre")

# 5-Jahres-Projektion
print("\n📊 5-JAHRES-PROJEKTION:")
print("-" * 50)
print(f"{'Jahr':<6} {'Cashflow':>12} {'Kumuliert':>12} {'ROI kum.':>12}")
print("-" * 50)

kumuliert = 0
for jahr in range(1, 6):
    if jahr == 1:
        cashflow = netto_einsparung - ki_investition
    else:
        cashflow = netto_einsparung
    kumuliert += cashflow
    roi_kum = kumuliert / ki_investition * 100
    print(f"{jahr:<6} {cashflow:>+12,.0f} € {kumuliert:>+12,.0f} € {roi_kum:>+11.0f}%")

print("\n💡 FAZIT:")
print(f"   Die KI-Investition amortisiert sich nach {payback_monate:.0f} Monaten.")
print(f"   Nach 5 Jahren beträgt der kumulierte ROI {roi_kum:.0f}%.")

### ✏️ Aufgabe 3: Use Case Entwicklung

Entwickeln Sie einen eigenen KI-Use-Case für die Produktion:

1. **Problemdefinition**: Welches Problem soll gelöst werden?
2. **Datenquellen**: Welche Daten werden benötigt?
3. **KI-Methode**: Welche Algorithmen sind geeignet?
4. **Nutzen**: Welcher Mehrwert wird geschaffen?
5. **Implementierung**: Wie könnte die Umsetzung aussehen?

In [ ]:
# Ihre Lösung für Aufgabe 3:
# ===========================

# TODO: Use Case dokumentieren
# Optional: Prototyp implementieren



In [ ]:
# ===================================================================
# LÖSUNG Aufgabe 3: Beispiel Use Case (zur Selbstkontrolle)
# ===================================================================

print("📝 LÖSUNG AUFGABE 3: Use Case - Predictive Maintenance")
print("=" * 70)

use_case = {
    'titel': 'Predictive Maintenance für CNC-Maschinen',
    'problem': '''
    Ungeplante Maschinenausfälle führen zu:
    • Produktionsstillständen (Ø 4h pro Ausfall)
    • Eilreparaturkosten (+50% Aufschlag)
    • Qualitätsproblemen vor Ausfall
    • Lieferverzögerungen und Konventionalstrafen
    
    Ziel: Ausfälle 24-72h im Voraus vorhersagen
    ''',
    'datenquellen': [
        'Vibrationssensoren (Beschleunigung, Frequenzspektrum)',
        'Temperatursensoren (Spindel, Kühlmittel)',
        'Stromaufnahme (Antriebe)',
        'Akustische Sensoren (Geräuschpegel)',
        'Wartungshistorie (SAP PM)',
        'Betriebsstunden und Lastzyklen'
    ],
    'ki_methoden': [
        'Anomalieerkennung: Isolation Forest, Autoencoder',
        'Zeitreihenanalyse: LSTM, Transformer',
        'Survival Analysis: Cox Regression',
        'Klassifikation: Random Forest, XGBoost'
    ],
    'nutzen': {
        'Ungeplante Stillstände': 'Mögliche Reduktion; im Pilot zu messen',
        'Wartungskosten': 'Mögliche Veränderung; gegen zusätzliche Systemkosten abzuwägen',
        'Anlagenverfügbarkeit': 'Mögliche Verbesserung; anhand definierter Kennzahlen prüfen',
        'Produktqualität': 'Mögliche Verbesserung; separat empirisch nachweisen'
    },
    'implementierung': [
        'Monat 1-2: Sensorik installieren, Datenerfassung starten',
        'Monat 3-4: Verfügbare Historie auswerten, neue Daten sammeln und EDA durchführen',
        'Monat 5-6: Erste ML-Modelle trainieren und evaluieren',
        'Monat 7-8: Dashboard entwickeln, Pilotbetrieb',
        'Monat 9-12: Rollout, Integration in Wartungsplanung'
    ]
}

print(
    "\nFiktiver Übungsfall: Ausfalldauer, Kostenaufschlag, "
    "Vorhersagehorizont und Zeitplan sind Annahmen."
)
print(
    "Ob Ausfälle 24–72 Stunden vorher erkennbar sind, "
    "muss für die konkrete Anlage und Fehlerart geprüft werden."
)


print(f"\n🎯 USE CASE: {use_case['titel']}")
print("\n📋 PROBLEMSTELLUNG:")
print(use_case['problem'])

print("\n📊 DATENQUELLEN:")
for source in use_case['datenquellen']:
    print(f"   • {source}")

print("\n🤖 KI-METHODEN:")
for method in use_case['ki_methoden']:
    print(f"   • {method}")

print("\n💰 MÖGLICHER NUTZEN – IM PILOT ZU PRÜFEN:")
for nutzen, wert in use_case['nutzen'].items():
    print(f"   • {nutzen}: {wert}")

print("\n📅 IMPLEMENTIERUNGSPLAN:")
for schritt in use_case['implementierung']:
    print(f"   • {schritt}")

print(
    "\nDie Datensammlung muss ausreichend relevante Ausfälle "
    "und Betriebszustände abdecken; dafür können längere "
    "Beobachtungszeiträume erforderlich sein."
)
print(
    "Anomalieerkennung allein liefert noch keine validierte "
    "Vorhersage des Ausfallzeitpunkts."
)

---

## 8. Zusammenfassung und Ausblick

### Was haben wir gelernt?

| Thema | Kernaussage |
|-------|-------------|
| **ML-Pipeline** | Vorverarbeitung und Modell gemeinsam fitten und anwenden |
| **Feature Engineering** | Fachlich begründete Merkmale als überprüfbare Hypothesen entwickeln |
| **Modellvergleich** | Auswahl durch CV auf Trainingsdaten; abschließende Bewertung auf Testdaten |
| **Feature Importance** | Modellbezogene prädiktive Relevanz, kein Kausalitätsnachweis |
| **Szenarioanalyse** | Modellprognosen für alternative Konfigurationen, keine garantierten Maßnahmenwirkungen |
| **Industrie 4.0** | Daten, Vernetzung und geeignete KI-Anwendungen gemeinsam betrachten |
| **Implementierung** | Fallspezifisch, schrittweise und mit Validierung, Monitoring und klaren Verantwortlichkeiten |

### Die Zukunft der intelligenten Produktion

Die folgende Darstellung unterscheidet Analysearten.
Sie ist **keine allgemeine zeitliche Abfolge**: Deskriptive, prädiktive und präskriptive Verfahren werden bereits heute eingesetzt und ergänzen sich.


```
       HEUTE                    MORGEN                   ÜBERMORGEN
    ┌──────────┐             ┌──────────┐             ┌──────────┐
    │Deskriptiv│   ────▶    │Prädiktiv │   ────▶     │Präskriptiv│
    │"Was ist  │             │"Was wird │             │"Was soll │
    │passiert?"│             │passieren?│             │ich tun?" │
    └──────────┘             └──────────┘             └──────────┘
    
    Dashboards              ML-Modelle              Autonome
    Reports                 Forecasting             Optimierung
    BI-Tools                Alerting                Selbststeuerung
```

In [ ]:
# ===================================================================
# Abschluss und Checkliste
# ===================================================================

print("\n" + "=" * 70)
print("📋 CHECKLISTE – KI-Integration in der Produktion")
print("=" * 70)

checkliste = [
    (
        "Produktionsdaten verstehen",
        "Datenqualität, Herkunft und vorgegebene Generatorbeziehungen prüfen"
    ),
    (
        "ML-Pipeline aufbauen",
        "One-Hot-Encoding und Modell innerhalb derselben Pipeline"
    ),
    (
        "Modelle auswählen",
        "Cross-Validation ausschließlich auf Trainingsdaten"
    ),
    (
        "Unabhängig bewerten",
        "Ausgewähltes Modell abschließend auf zurückgehaltenen Testdaten"
    ),
    (
        "Feature Importance interpretieren",
        "Prädiktive Relevanz, Verzerrungen und fehlenden Kausalitätsnachweis beachten"
    ),
    (
        "Szenarien vergleichen",
        "Modellprognosen, Datenabdeckung und technische Machbarkeit unterscheiden"
    ),
    (
        "Implementierung planen",
        "Validierung, Monitoring, Verantwortlichkeiten und Einsatzgrenzen"
    ),
    (
        "Wirtschaftlichkeit prüfen",
        "Annahmen, Nettoeinsparung und einfache Amortisation transparent machen"
    )
]

for i, (title, explanation) in enumerate(checkliste, 1):
    print(f"\n☐ {i}. {title}")
    print(f"   → {explanation}")

print("\n" + "=" * 70)
print("🎓 ABSCHLUSS")
print("=" * 70)
print(
    "Sie können nun einen einfachen ML-Workflow nachvollziehen "
    "und zentrale Anforderungen an einen praktischen Einsatz benennen."
)
print(
    "Das Notebook demonstriert Modellierung und Analyse, "
    "aber keine vollständige produktive KI-Integration."
)